# Pandas : limites et écosystèmes alternatifs

Lino Galiana  
2026-07-20

<div class="badge-container"><div class="badge-text">Pour essayer les exemples présents dans ce tutoriel :</div><a href="https://github.com/linogaliana/python-datascientist-notebooks/blob/main/notebooks/manipulation/02_pandas_beyond.ipynb" class="badge" target="_blank" rel="noopener"><img src="https://img.shields.io/static/v1?logo=github&label=&message=View%20on%20GitHub&color=181717" alt="View on GitHub"></a>
<a href="https://datalab.sspcloud.fr/launcher/ide/vscode-python?autoLaunch=true&name=«02_pandas_beyond»&init.personalInit=«https%3A%2F%2Fraw.githubusercontent.com%2Flinogaliana%2Fpython-datascientist%2Fmain%2Fsspcloud%2Finit-vscode.sh»&init.personalInitArgs=«manipulation%2002_pandas_beyond»" class="badge" target="_blank" rel="noopener"><img src="https://custom-icon-badges.demolab.com/badge/SSP%20Cloud-Lancer_avec_VSCode-blue?logo=vsc&logoColor=white" alt="Onyxia"></a>
<a href="https://datalab.sspcloud.fr/launcher/ide/jupyter-python?autoLaunch=true&name=«02_pandas_beyond»&init.personalInit=«https%3A%2F%2Fraw.githubusercontent.com%2Flinogaliana%2Fpython-datascientist%2Fmain%2Fsspcloud%2Finit-jupyter.sh»&init.personalInitArgs=«manipulation%2002_pandas_beyond»" class="badge" target="_blank" rel="noopener"><img src="https://img.shields.io/badge/SSP%20Cloud-Lancer_avec_Jupyter-orange?logo=Jupyter&logoColor=orange" alt="Onyxia"></a>
<a href="https://colab.research.google.com/github/linogaliana/python-datascientist-notebooks-colab//blob/main//notebooks/manipulation/02_pandas_beyond.ipynb" class="badge" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<a href="https://codespaces.new/linogaliana/python-datascientist-notebooks?quickstart=1&ref=main" class="badge" target="_blank" rel="noopener"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces"></a><br></div>

> **None**
>
> <div class="callout callout-style-default callout-note callout-titled">
> <div class="callout-header d-flex align-content-center">
> <div class="callout-icon-container">
> <i class="callout-icon"></i>
> </div>
> <div class="callout-title-container flex-fill">
> Note
> </div>
> </div>
> <div class="callout-body-container callout-body">
>
> Ceci est la version française 🇫🇷 de ce chapitre, pour voir la version anglaise rendez-vous sur <a href="https://pythonds.linogaliana.fr//home/runner/work/python-datascientist/python-datascientist/en/content/manipulation/02_pandas_beyond.qmd">le site du cours</a>.
>
> </div>
> </div>


<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Compétences à l’issue de ce chapitre
</div>
</div>
<div class="callout-body-container callout-body">

-   Connaître les principales limites de la syntaxe de `Pandas` ;
-   Savoir chaîner des opérations de manière plus lisible grâce aux *pipes* ;
-   Connaître les principaux *packages* alternatifs à `Pandas` (`Polars`, `DuckDB`, `Spark`).

</div>
</div>

# 1. Introduction

Les deux chapitres précédents ont permis d’explorer en profondeur la richesse de l’écosystème `Pandas`, qu’il s’agisse [d’associer des données grâce à des jointures](../../content/manipulation/02_pandas_joins.qmd) ou de [construire des statistiques descriptives et de restructurer des données](../../content/manipulation/02_pandas_stats.qmd). `Pandas` est un indispensable dans la boite à outil du *data scientist*.

Ce court chapitre propose de prendre un peu de recul. Malgré toutes ses qualités, `Pandas` n’est pas parfait: sa syntaxe, héritée de plus de quinze ans d’histoire, comporte quelques aspérités. Nous verrons également que d’autres paradigmes, plus récents, permettent d’aller au-delà de `Pandas`, notamment lorsqu’il s’agit de traiter des volumes de données plus importants.

## 1.1 Données de ce chapitre

Pour illustrer ce chapitre, nous repartons du jeu de données brut des émissions de gaz à effet de serre de l’Ademe, déjà utilisé dans les chapitres précédents:


In [ ]:
import numpy as np
import pandas as pd

url = "https://data.ademe.fr/data-fair/api/v1/datasets/igt-pouvoir-de-rechauffement-global/convert"
emissions = pd.read_csv(url)
emissions['dep'] = emissions["INSEE commune"].str[:2]

Pour obtenir des résultats reproductibles, on peut fixer la racine du générateur
pseudo-aléatoire.


In [ ]:
np.random.seed(123)

# 2. `Pandas` dans une chaine d’opérations

En général, dans un projet, le nettoyage de données va consister en un ensemble de
méthodes appliquées à un `DataFrame` ou alors une `Serie` lorsqu’on travaille exclusivement sur une colonne.
Autrement dit, ce qui est généralement attendu lorsqu’on fait du `Pandas` c’est d’avoir une chaîne qui prend un `DataFrame` en entrée et ressort ce même `DataFrame` enrichi, ou une version agrégée de celui-ci, en sortie.

Cette manière de procéder est le coeur de la syntaxe `dplyr` en `R` mais n’est pas forcément native en `Pandas` selon les opérations qu’on désire mettre en oeuvre. En effet, la manière naturelle de mettre à jour un *dataframe* en `Pandas` passe souvent par une syntaxe du type:


In [ ]:
import numpy as np
import pandas as pd

data = [[8000, 1000], [9500, np.nan], [5000, 2000]]
df = pd.DataFrame(data, columns=['salaire', 'autre_info'])
df['salaire_net'] = df['salaire']*0.8

En `SQL` on pourrait directement mettre à jour notre base de données avec la nouvelle colonne :

``` sql
SELECT *, salaire*0.8 AS salaire_net FROM df
```

L’écosystème du *tidyverse* en `R`, l’équivalent de `Pandas`, fonctionne selon la même logique que SQL de mise à jour de table. On ferait en effet la commande suivante avec `dplyr`:

``` r
df %>% mutate(salaire_net = salaire*0.8)
```

Techniquement on pourrait faire ceci avec un `assign` en `Pandas`


In [ ]:
df = df.drop("salaire_net", axis = "columns")
df = df.assign(salaire_net = lambda s: s['salaire']*0.8)

Cependant cette syntaxe `assign` n’est pas très naturelle. Il est nécessaire de lui passer une *lambda function* qui attend comme *input* un `DataFrame` là où on voudrait une colonne. Il ne s’agit donc pas vraiment d’une syntaxe lisible et pratique.

Il est néanmoins possible d’enchaîner des opérations sur des jeux de données grâce aux [*pipes*](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html). Ceux-ci reprennent la même philosophie que celle de `dplyr`, elle-même inspirée du *pipe* Linux.
Cette approche permettra de rendre plus lisible le code en définissant des fonctions effectuant des opérations sur une ou plusieurs colonnes d’un DataFrame. Le premier argument à indiquer à la fonction est le `DataFrame`, les autres sont ceux permettant de contrôler son comportement


In [ ]:
def calcul_salaire_net(df: pd.DataFrame, col: str, taux: float = 0.8):
  df["salaire_net"] = df[col]*taux
  return df

Ce qui transforme notre chaine de production en


In [ ]:
(
  df
  .pipe(calcul_salaire_net, "salaire")
)

## 2.1 Quelques limites sur la syntaxe de `Pandas`

Il y a un avant et un après `Pandas` dans l’analyse de données en `Python`. Sans ce *package* ô combien pratique `Python`, malgré toutes les forces de ce langage, aurait eu du mal à s’installer dans le paysage de l’analyse de données. Cependant, si `Pandas` propose une syntaxe cohérente sur de nombreux aspects, elle n’est pas parfaite non plus. Les paradigmes plus récents d’analyse de données en `Python` ont d’ailleurs parfois l’ambition de corriger ces imperfections syntaxiques là.

Parmi les points les plus génants au quoditien il y a le besoin de régulièrement faire des `reset_index` lorsqu’on construit des statistiques descriptives. En effet, il peut être dangereux de garder des indices qu’on ne contrôle pas bien car, sans attention de notre part lors des phases de *merge*, ils peuvent être utilisés à mauvais escient par `Pandas` pour joindre les données ce qui peut provoquer des suprises.

`Pandas` est extrêmement bien fait pour restructurer des données du format *long* to *wide* ou *wide* to *long*. Cependant, ce n’est pas la seule manière de restructurer un jeu de données qu’on peut vouloir mettre en oeuvre. Il arrive régulièrement qu’on désire comparer la valeur d’une observation à celle d’un groupe à laquelle elle appartient. C’est notamment particulièrement utile dans une phase d’analyse des anomalies, valeurs aberrantes ou lors d’une investigation de détection de fraude. De manière native, en `Pandas`, il faut construire une statistique agrégée par groupe et refaire un *merge* aux données initiales par le biais de la variable de groupe. C’est un petit peu fastidieux:


In [ ]:
emissions_moyennes = emissions.groupby("dep").agg({"Agriculture": "mean"}).reset_index()
emissions_enrichies = (
  emissions
  .merge(emissions_moyennes, on = "dep", suffixes = ['', '_moyenne_dep'])
)
emissions_enrichies['relatives'] = emissions_enrichies["Agriculture"]/emissions_enrichies["Agriculture_moyenne_dep"]
emissions_enrichies.head()

Dans le *tidyverse*, cette opération en deux temps pourrait être faite en une seule étape, ce qui est plus pratique

``` r
emissions %>%
  group_by(dep) %>%
  mutate(relatives = Agriculture/mean(Agriculture))
```

Ce n’est pas si grave mais cela alourdit la longueur des chaines de traitement faites en `Pandas` et donc la charge de maintenance pour les faire durer dans le temps.

De manière plus générale, les chaînes de traitement `Pandas` peuvent être assez verbeuses, car il faut régulièrement redéfinir le `DataFrame` qu’on utilise plutôt que simplement les colonnes. Par exemple, pour faire un filtre sur les lignes et les colonnes, il faudra faire:


In [ ]:
(
  emissions
  .loc[
    (emissions["dep"] == "12") & (emissions["Routier"]>500), ['INSEE commune', 'Commune']
  ]
  .head(5)
)

En SQL on pourrait se contenter de faire référence aux colonnes dans le filter

``` sql
SELECT "INSEE commune", 'Commune'
FROM emissions
WHERE dep=="12" AND Routier>500
```

Dans le *tidyverse* (`R`) on pourrait aussi faire ceci simplement

``` r
df %>%
  filter(dep=="12", Routier>500) %>%
  select(`INSEE commune`, `Commune`)
```

# 3. Les autres paradigmes

Malgré toutes les limites que nous venons d’évoquer, et les solutions alternatives que nous allons présenter, `Pandas` reste LE *package* central de l’écosystème de la donnée avec `Python`. Nous allons voir dans les prochains chapitres son intégration native à l’écosystème `Scikit` pour le *machine learning* ou l’extension de `Pandas` aux données spatiales avec `GeoPandas`.

Les autres solutions techniques que nous allons ici évoquer peuvent être pertinentes si on désire traiter des volumes de données importants ou si on désire utiliser des syntaxes alternatives.

Les principales alternatives à `Pandas` sont [`Polars`](https://pola.rs/), [`DuckDB`](https://duckdb.org/) et [`Spark`](https://spark.apache.org/docs/latest/api/python/index.html). Il existe également [`Dask`](https://www.dask.org/), une librairie pour paralléliser des traitements écris en `Pandas`.

## 3.1 `Polars`

`Polars` est certainement le paradigme le plus inspiré de `Pandas`, jusqu’au choix du nom. La première différence fondamentale est dans les couches internes utilisées. `Polars` s’appuie sur l’implémentation `Rust` de `Arrow` là où `Pandas` s’appuie sur `Numpy` ce qui est facteur de perte de performance. Cela permet à `Polars` d’être plus efficace sur de gros volumes de données, d’autant que de nombreuses opérations sont parallélisées et reposent sur l’évaluation différées (*lazy evaluation*) un principe de programmation qui permet d’optimiser les requêtes pour ne pas les exécuter dans l’ordre de définition mais dans un ordre logique plus optimal.

Une autre force de `Polars` est la syntaxe plus cohérente, qui bénéficie du recul d’une quinzaine d’années d’existence de `Pandas` et d’une petite dizaine d’années de `dplyr` (le *package* de manipulation de données au sein du paradigme du *tidyverse* en `R`). Pour reprendre l’exemple précédent, il n’est plus nécessaire de forcer la référence au *DataFrame*, dans une chaîne d’exécution toutes les références ultérieures seront faites au regard du *DataFrame* de départ


In [ ]:
import polars as pl
emissions_polars = pl.from_pandas(emissions)
(
  emissions_polars
  .filter(pl.col("dep") == "12", pl.col("Routier") > 500)
  .select('INSEE commune', 'Commune')
  .head(5)
)

Pour découvrir `Polars`, de nombreuses ressources en ligne sont accessibles, notamment [ce *notebook*](https://github.com/InseeFrLab/ssphub/blob/main/post/polars/polars-tuto.ipynb) construit pour le réseau des *data scientists* de la statistique publique.

## 3.2 `DuckDB`

DuckDB est le nouveau venu dans l’écosystème de l’analyse de données repoussant les limites des données pouvant être traitées avec `Python` sans passer par des outils *big data* comme `Spark`.
DuckDB est la quintessence d’un nouveau paradigme, celui du [*“Big data is dead”*](https://motherduck.com/blog/big-data-is-dead/), où on peut traiter des données de volumétrie importante sans recourir à des infrastructures imposantes.

Outre sa grande efficacité, puisqu’avec DuckDB on peut traiter des données d’une volumétrie supérieure à la mémoire vive de l’ordinateur ou du serveur, DuckDB présente l’avantage de proposer une syntaxe uniforme quelle que soit le langage qui appelle DuckDB (`Python`, `R`, `C++` ou `Javascript`). DuckDB privilégie la syntaxe SQL pour traiter les données avec de nombreuses fonctions pré-implementées pour simplifier certaines transformations de données (par exemple pour les [données textuelles](https://duckdb.org/docs/sql/functions/char.html), les [données temporelles](https://duckdb.org/docs/sql/functions/time), etc.).

Par rapport à d’autres systèmes s’appuyant sur SQL, comme [`PostGreSQL`](https://www.postgresql.org/), `DuckDB` est très simple d’installation, ce n’est qu’une librairie `Python` là où beaucoup d’outils comme `PostGreSQL` nécessite une infrastructure adaptée.

Pour reprendre l’exemple précédent, on peut utiliser directement le code SQL précédent


In [ ]:
import duckdb
duckdb.sql(
  """
  SELECT "INSEE commune", "Commune"
  FROM emissions
  WHERE dep=='12' AND Routier>500
  LIMIT 5
  """)

┌───────────────┬─────────────────────┐
│ INSEE commune │       Commune       │
│    varchar    │       varchar       │
├───────────────┼─────────────────────┤
│ 12001         │ AGEN-D'AVEYRON      │
│ 12002         │ AGUESSAC            │
│ 12006         │ ALRANCE             │
│ 12007         │ AMBEYRAC            │
│ 12008         │ ANGLARS-SAINT-FELIX │
└───────────────┴─────────────────────┘

Ici la clause `FROM emissions` vient du fait qu’on peut directement exécuter du SQL depuis un objet `Pandas` par le biais de `DuckDB`. Si on fait la lecture directement dans la requête, celle-ci se complexifie un petit peu mais la logique est la même


In [ ]:
import duckdb
duckdb.sql(
  f"""
  SELECT "INSEE commune", "Commune"
  FROM read_csv_auto("{url}")
  WHERE
    substring("INSEE commune",1,2)=='12'
    AND
    Routier>500
  LIMIT 5
  """)

┌───────────────┬─────────────────────┐
│ INSEE commune │       Commune       │
│    varchar    │       varchar       │
├───────────────┼─────────────────────┤
│ 12001         │ AGEN-D'AVEYRON      │
│ 12002         │ AGUESSAC            │
│ 12006         │ ALRANCE             │
│ 12007         │ AMBEYRAC            │
│ 12008         │ ANGLARS-SAINT-FELIX │
└───────────────┴─────────────────────┘

Le rendu du *DataFrame* est légèrement différent de `Pandas` car, comme `Polars` et de nombreux systèmes de traitement de données volumineuses, `DuckDB` repose sur l’évaluation différée et donc ne présente en *display* qu’un échantillon de données.
`DuckDB` et `Polars` sont d’ailleurs très bien intégrés l’un à l’autre. On peut très bien faire du SQL sur un objet `Polars` via `DuckDB` ou appliquer des fonctions `Polars` sur un objet initialement lu avec `DuckDB`.

L’un des intérêts de `DuckDB` est son excellente intégration avec l’écosystème `Parquet`, le format de données déjà mentionné qui devient un standard dans le partage de données (il s’agit, par exemple, de la pierre angulaire du partage de données sur la plateforme *HuggingFace*). Pour en savoir plus sur `DuckDB` et découvrir son intérêt pour lire les données du recensement de la population française, vous pouvez consulter [ce post de blog](https://ssphub.netlify.app/post/parquetrp/).

## 3.3 `Spark` et le *big data*

`DuckDB` a repoussé les frontières du *big data* qu’on peut définir comme le volume de données à partir duquel on ne peut plus traiter celles-ci sur une machine sans mettre en oeuvre une stratégie de parallélisation.

Néanmoins, pour les données très volumineuses, `Python` est très bien armé grâce à la librairie [`PySpark`](https://spark.apache.org/docs/latest/api/python/index.html). Celle-ci est une API en Python pour le langage `Spark`, un langage *big data* basé sur Scala. Ce paradigme est construit sur l’idée que les utilisateurs de `Python` y accèdent par le biais de *cluster* avec de nombreux noeuds pour traiter la donnée de manière parallèle. Celle-ci sera lue par blocs, qui seront traités en parallèle en fonction du nombre de noeuds parallèles. L’API DataFrame de `Spark` présente une syntaxe proche de celle des paradigmes précédents avec une ingénieurie plus complexe en arrière-plan liée à la parallélisation native.
